In [1]:
from pathlib import Path
import sys,json,hashlib,importlib.util,zipfile
import pandas as pd
ROOT=Path("/home/featurize/work/biological_prior_validation").resolve()
RUN=ROOT/"data/splits/step2_q05_seed20261008"
PROTOCOL_NAME="task2_five_methods"
if not ROOT.is_dir():raise FileNotFoundError(f"Please modify ROOT：{ROOT}")


In [2]:
def load_module(name,path):
    spec=importlib.util.spec_from_file_location(name,path)
    module=importlib.util.module_from_spec(spec);sys.modules[name]=module;spec.loader.exec_module(module)
    return module
mc_path=ROOT/"src/randomized_controls.py"
if not mc_path.exists():raise FileNotFoundError("First run 09_task2_step1_five_methods.ipynb")
mc=load_module("randomized_controls",mc_path)
old_identity=mc.js(RUN/"task2_protocols"/PROTOCOL_NAME/"protocol_identity.json")
assert mc.ENGINE_SHA==old_identity["engine_sha256"],"2.1 Source Code Identity Mismatch"
print("2.1 source verified:",mc.ENGINE_SHA)


2.1 source verified: 3712e462217762223b90f6533e1260fb4a1d13c603ec570d5e01f53bedcaa6bf


In [3]:
SOURCE = '"""Task 2.2: frozen development-only schedules and audited base normalization.\nNo real fits or target access in prepare(); real precision gates belong to 2.3.\n"""\nfrom pathlib import Path\nimport json, hashlib, itertools, math, shutil\nimport numpy as np\nimport pandas as pd\nimport scipy.sparse as sp\nfrom scipy.sparse.linalg import LinearOperator, cg\n\nENGINE_SHA=hashlib.sha256(Path(__file__).read_bytes()).hexdigest()\ndef js(p): return json.loads(Path(p).read_text())\ndef sha(p): return hashlib.sha256(Path(p).read_bytes()).hexdigest()\ndef save(p,obj):\n p=Path(p);p.parent.mkdir(parents=True,exist_ok=True)\n t=p.with_suffix(p.suffix+\'.tmp\');t.write_text(json.dumps(obj,indent=2,ensure_ascii=False,allow_nan=False));t.replace(p)\ndef freeze(p,obj):\n if Path(p).exists() and js(p)!=obj: raise ValueError(f\'Frozen rules changed: {p}\')\n save(p,obj)\ndef key(x): return format(float(x),\'.17g\')\n\ndef fine_axis(coarse,center):\n """Five fresh positive points plus zero, within original positive bounds."""\n a=np.asarray(coarse,float)\n if a.ndim!=1 or len(a)<3 or not np.isfinite(a).all() or not np.array_equal(a,np.unique(a)) or a[0]!=0 or (a<0).any(): raise ValueError(\'Sorted coarse axis including zero and two positives required\')\n if center not in a: raise ValueError(\'Center must be a coarse value\')\n lo,hi=np.log10(a[1]),np.log10(a[-1]);width=min(1. if center==0 else 2.,hi-lo)\n middle=lo+width/2 if center==0 else np.log10(center)\n left=float(np.clip(middle-width/2,lo,hi-width));right=left+width\n pool=10**np.linspace(left,right,2003)[1:-1]\n pool=pool[~np.any(np.isclose(pool[:,None],a[None,:],rtol=1e-12,atol=0),axis=1)]\n if len(pool)<5: raise ValueError(\'Positive range too narrow for refinement\')\n indices=np.round(np.linspace(1/6,5/6,5)*(len(pool)-1)).astype(int)\n out=[0.]+pool[indices].tolist()\n if len(set(out))!=6: raise ValueError(\'Degenerate fine grid\')\n return out\n\ndef make_grid(coarse,common,representation=None):\n a=coarse[\'alpha\'];b=coarse[\'beta\'];lr=coarse[\'lambda_rules\']\n if len(lr[\'quantiles\'])!=5: raise ValueError(\'Expected audited five lambda quantiles\')\n return {\n \'schema\':\'task2-parameter-grid-v1\',\'status\':\'complete_search_rules_frozen_BEFORE_REAL_BENCHMARK\',\n \'scenario\':\'A\',\'source_coarse\':coarse,\n \'coarse\':{\'alpha\':a,\'beta\':b,\'lambda_rules\':[f\'q{q:g}\' for q in lr[\'quantiles\']]+[\'high\',\'fixed_limit\']},\n \'fine\':{\'rounds\':1,\'positive_points_per_axis\':5,\'zero_in_every_axis\':True,\n  \'alpha_lookup\':{key(v):fine_axis(a,v) for v in a},\'beta_lookup\':{key(v):fine_axis(b,v) for v in b},\n  \'window_log10_width\':{\'positive_center\':2.,\'zero_center\':1.},\n  \'bounds\':\'same positive coarse bounds for every method; clipped window shifted inward, never expanded from B\',\n  \'fresh_points\':\'positive fine values avoid coarse values; (alpha=0,beta=0) is cached, not fitted twice\',\n  \'lambda\':{\'finite_best\':\'six multipliers times training-only threshold of selected coarse lambda rule, recomputed for each fine alpha\',\n   \'multipliers\':[.25,.5,.75,1.,1.5,2.],\'fallback_if_beta0_or_fixed\':\'same seven coarse rules\',\n   \'include_fixed_limit\':True,\'degenerate_energy\':\'only fixed_limit, report effective budget reduction\'}},\n \'selection\':{\'metric\':\'Step3 primary mean Poisson deviance on common development V pairs, incl zeros\',\n  \'role\':\'development_only\',\'cross_validation\':\'Step4 fixed T/V split, no additional target-dependent split\',\n  \'winner\':\'best valid candidate over coarse PLUS fine; coarse winner retained\',\n  \'ties\':\'exact score, alpha, beta, fixed_limit first, predefined lambda rule order\',\n  \'failures\':\'retain failed rows/status; fail scope if all coarse candidates invalid; no adaptive seed replacement\',\n  \'boundary\':\'record coarse/final boundary flags; final boundary => report limitation, no automatic extra rounds\'},\n \'methods\':{\'M0\':\'coarse alpha then own fine alpha; beta=0\',\n  \'M1\':\'joint alpha x beta coarse; own winning region, joint fine\',\n  \'M2\':\'same joint alpha/beta rules plus lambda, own winning region\',\n  \'M3\':\'no search; inherits new Task2 M2 parameters; all permutations reported\',\n  \'M4\':\'each seed independently uses exact M1 ranges, fine algorithm and budgets\'},\n \'normalization\':coarse[\'normalization\'],\'common_protocol_sha256\':None,\n \'shared_representation\':representation or {\'status\':\'synthetic fixture only\'},\n \'scoring\':common[\'scoring\'],\n \'budget\':{\'M0\':{\'coarse\':len(a),\'fine_slots\':6,\'new_fine_fits\':5,\'nominal_unique\':len(a)+5},\n  \'M1\':{\'coarse\':len(a)*len(b),\'fine_slots\':36,\'new_fine_fits\':35,\'nominal_unique\':len(a)*len(b)+35},\n  \'M2\':{\'coarse\':len(a)*(1+(len(b)-1)*7),\'fine_slots\':216,\'new_fine_fits\':215,\'nominal_unique\':len(a)*(1+(len(b)-1)*7)+215},\n  \'M3\':{\'coarse\':0,\'fine_slots\':0,\'new_fine_fits\':0,\'nominal_unique\':0},\n  \'M4\':{\'coarse\':len(a)*len(b),\'fine_slots\':36,\'new_fine_fits\':35,\'nominal_unique\':len(a)*len(b)+35},\n  \'meaning\':\'M1/M4 equal candidate-attempt budget per scope/seed; not equal wall time. M2 nominal maximum; degenerate/equal thresholds may share fits, never add unplanned candidates.\'},\n \'precision\':{\'standard\':coarse[\'convergence\'],\n  \'tight\':{\'cg_rtol\':1e-10,\'cg_maxiter\':3000,\'adaptive_max_outer\':300,\'adaptive_x_rtol\':1e-9,\'adaptive_s_atol\':1e-8},\n  \'thresholds\':{\'relative_prediction_l2\':1e-6,\'relative_development_deviance\':1e-7,\'absolute_s_max\':1e-5},\n  \'probe_parameters\':[{\'alpha\':0.,\'beta\':0.},{\'alpha\':a[1],\'beta\':b[1]},{\'alpha\':1.,\'beta\':1.},{\'alpha\':a[-1],\'beta\':b[-1]}],\n  \'probes\':\'per first-four context/label: M0(beta forced 0)/M1/M2/M4(seed0), explicit probe parameters; M2 q50/high and identical deterministic initialization; fixed T\',\n  \'failure\':\'stop before search; numerical-only amendment with new version, no final B use\',\n  \'gate\':\'synthetic tests here are not a real-data precision PASS; real T/V precision checks mandatory in 2.3 before search\'},\n \'new_B_read\':False,\'real_search_run\':False,\'scenario_B\':\'BLOCKED_PENDING_REPLICATE_PROVENANCE\'}\n\ndef candidate_id(c):\n return \'|\'.join([key(c[\'alpha\']),key(c[\'beta\']),c.get(\'lambda_rule\',\'fixed_limit\')])\ndef candidates(method,grid,phase=\'coarse\',winner=None):\n if method==\'M3\': return []\n if method not in [\'M0\',\'M1\',\'M2\',\'M4\'] or phase not in [\'coarse\',\'fine\']: raise ValueError(\'Invalid method/phase\')\n if phase==\'coarse\':a=grid[\'coarse\'][\'alpha\'];b=grid[\'coarse\'][\'beta\']\n else:\n  if winner is None: raise ValueError(\'Coarse winner required\')\n  a=grid[\'fine\'][\'alpha_lookup\'][key(winner[\'alpha\'])];b=grid[\'fine\'][\'beta_lookup\'][key(winner[\'beta\'])]\n if method==\'M0\': b=[0.]\n out=[]\n for av,bv in itertools.product(a,b):\n  rules=[\'fixed_limit\']\n  if method==\'M2\' and bv>0:\n   if phase==\'fine\' and winner[\'beta\']>0 and winner[\'lambda_rule\']!=\'fixed_limit\':\n    rules=[f"scaled:{winner[\'lambda_rule\']}:{key(m)}" for m in grid[\'fine\'][\'lambda\'][\'multipliers\']]+[\'fixed_limit\']\n   else:rules=grid[\'coarse\'][\'lambda_rules\']\n  for rule in rules:out.append({\'alpha\':float(av),\'beta\':float(bv),\'lambda_rule\':rule})\n assert len({candidate_id(c) for c in out})==len(out)\n return out\n\ndef winner(rows,expected):\n ids=[candidate_id(r) for r in rows]\n if set(ids)!={candidate_id(c) for c in expected} or len(ids)!=len(set(ids)):raise ValueError(\'Incomplete or duplicate search table\')\n valid=[]\n for r in rows:\n  if r.get(\'status\')==\'OK\':\n   if r.get(\'target_role\')!=\'development_only\': raise ValueError(\'Selection requires development-only role\')\n   value=r.get(\'development_deviance\')\n   if value is None or not np.isfinite(value) or value<0: raise ValueError(\'Invalid deviance\')\n   valid.append(r)\n if not valid: raise ValueError(\'No valid candidate\')\n order={candidate_id(c):i for i,c in enumerate(expected)}\n return min(valid,key=lambda r:(r[\'development_deviance\'],r[\'alpha\'],r[\'beta\'],r[\'lambda_rule\']!=\'fixed_limit\',order[candidate_id(r)]))\n\ndef search(method,grid,fit_score):\n """Future development-only executor. Callback accepts candidate, returns status/score.\n Returns complete rows, preserving failures and cache reuse. Not called on real data here.\n """\n if method==\'M3\':raise ValueError(\'M3 inherits M2, no parameter search\')\n cache={};rows=[]\n def run(items,phase):\n  for c in items:\n   cid=candidate_id(c);cached=cid in cache\n   if not cached:\n    try:result=fit_score(dict(c))\n    except (RuntimeError,ValueError,FloatingPointError) as e:result={\'status\':\'FAILED\',\'failure_reason\':str(e),\'development_deviance\':None,\'target_role\':\'development_only\'}\n    if result.get(\'status\') not in [\'OK\',\'FAILED\',\'SKIPPED_DEGENERATE\']:raise ValueError(\'Unknown fit status\')\n    cache[cid]={**c,**{k:v for k,v in result.items() if k not in c}}\n   rows.append({**cache[cid],\'phase\':phase,\'cache_reuse\':cached})\n coarse=candidates(method,grid);run(coarse,\'coarse\');best=winner(rows,coarse)\n fine=candidates(method,grid,\'fine\',best);run(fine,\'fine\')\n union={candidate_id(r):r for r in rows};final=winner(list(union.values()),list(union.values()))\n def flags(r):\n  return {axis:{\'zero\':r[axis]==0,\'positive_lower\':r[axis]==grid[\'coarse\'][axis][1],\'positive_upper\':r[axis]==grid[\'coarse\'][axis][-1]} for axis in [\'alpha\',\'beta\']}\n return {\'selected\':final,\'rows\':rows,\'unique_attempts\':len(cache),\'planned_slots\':len(rows),\'coarse_boundary_flags\':flags(best),\'final_boundary_flags\':flags(final)}\n\ndef threshold(rule,positive_training_energies,lr):\n e=np.asarray(positive_training_energies,float)\n if e.ndim!=1 or not np.isfinite(e).all() or (e<0).any():raise ValueError(\'Invalid training energies\')\n if rule==\'fixed_limit\':return \'fixed_limit\'\n e=e[e>0]\n if not len(e):return \'DEGENERATE_NO_POSITIVE_ENERGY\'\n if rule.startswith(\'scaled:\'):\n  _,anchor,m=rule.split(\':\');v=threshold(anchor,e,lr);return float(v)*float(m)\n if rule==\'high\':return float(lr[\'high_multiplier\']*np.quantile(e,.99))\n if rule.startswith(\'q\'):\n  q=float(rule[1:])\n  if q not in lr[\'quantiles\']:raise ValueError(\'Unfrozen quantile\')\n  return float(np.quantile(e,q))\n raise ValueError(\'Unknown threshold rule\')\n\ndef solve(C,lc,L,alpha,beta,rtol=1e-8,maxiter=1000):\n C=np.asarray(C,float);lc=np.asarray(lc,float);L=sp.csr_matrix(L)\n if C.ndim!=2 or min(C.shape)<1 or lc.shape!=(C.shape[0],) or L.shape!=(C.shape[1],C.shape[1]) or not np.isfinite(C).all() or not np.isfinite(lc).all() or (lc<0).any() or not np.isfinite(L.data).all():raise ValueError(\'Invalid operator/input\')\n if not np.isfinite(rtol) or rtol<=0 or not isinstance(maxiter,int) or maxiter<1:raise ValueError(\'Invalid solver controls\')\n off=L-sp.diags(L.diagonal());diff=L-L.T\n if (diff.nnz and np.max(np.abs(diff.data))>1e-10) or np.max(np.abs(np.asarray(L.sum(1)).ravel()))>1e-8 or (off.nnz and off.data.max()>1e-10) or (L.diagonal()<0).any():raise ValueError(\'Symmetric nonnegative-weight gene Laplacian required\')\n if not np.isfinite([alpha,beta]).all() or min(alpha,beta)<0:raise ValueError(\'Invalid parameters\')\n a=1+alpha*lc;diag=a[:,None]+beta*L.diagonal()[None,:]\n def mv(z):\n  x=z.reshape(C.shape);return (a[:,None]*x+beta*(L@x.T).T).ravel()\n op=LinearOperator((C.size,C.size),matvec=mv,dtype=float)\n pre=LinearOperator(op.shape,matvec=lambda z:z/diag.ravel(),dtype=float)\n x,info=cg(op,C.ravel(),M=pre,rtol=rtol,atol=0,maxiter=maxiter)\n residual=float(np.linalg.norm(mv(x)-C.ravel())/max(np.linalg.norm(C),1e-12))\n if info!=0 or residual>max(5*rtol,1e-12):raise RuntimeError(f\'CG failed: {info=}, {residual=}\')\n return x.reshape(C.shape),residual\n\ndef precision_comparison(standard,tight,loss,thresholds,s_standard=None,s_tight=None):\n a=np.asarray(standard,float);b=np.asarray(tight,float)\n if a.shape!=b.shape or not np.isfinite(a).all() or not np.isfinite(b).all():raise ValueError(\'Invalid prediction comparison\')\n x=float(np.linalg.norm(a-b)/max(np.linalg.norm(b),1e-12));da=float(loss(a));db=float(loss(b))\n if not np.isfinite([da,db]).all() or min(da,db)<0:raise ValueError(\'Invalid development loss\')\n d=abs(da-db)/max(abs(db),1e-12);s=None\n if (s_standard is None)!=(s_tight is None):raise ValueError(\'Both reliability vectors required\')\n if s_standard is not None:\n  sa=np.asarray(s_standard,float);sb=np.asarray(s_tight,float)\n  if sa.shape!=sb.shape or not np.isfinite(sa).all() or not np.isfinite(sb).all():raise ValueError(\'Invalid s comparison\')\n  s=float(np.max(np.abs(sa-sb),initial=0))\n return {\'status\':\'PASS\' if x<=thresholds[\'relative_prediction_l2\'] and d<=thresholds[\'relative_development_deviance\'] and (s is None or s<=thresholds[\'absolute_s_max\']) else \'FAIL\',\n \'relative_prediction_l2\':x,\'relative_development_deviance\':d,\'absolute_s_max\':s,\'standard_deviance\':da,\'tight_deviance\':db}\n\ndef prepare(root,run,protocol_name,mc,do_graph_audit=True):\n root=Path(root).resolve();run=Path(run).resolve()\n if not run.is_relative_to(root/\'data/splits\'):raise ValueError(\'Run outside project\')\n old=run/\'task2_protocols\'/protocol_name;identity=js(old/\'protocol_identity.json\');common=js(old/\'common_protocol.json\');coarse=js(old/\'parameter_grid_coarse.json\')\n for r in js(old/\'output_manifest.json\')[\'files\']:\n  if sha(old/r[\'name\'])!=r[\'sha256\']:raise ValueError(\'Step2.1 manifest mismatch: \'+r[\'name\'])\n if mc.ENGINE_SHA!=identity[\'engine_sha256\']:raise ValueError(\'Randomization source changed\')\n lock=run/\'evaluation/target_access_lock.json\';oldlock=sha(lock)\n if oldlock!=identity[\'target_history\'][\'target_access_lock_sha256\']:raise ValueError(\'Step5 lock changed\')\n if sha(Path(common[\'metric_contract\']))!=identity[\'references\'][\'metric_sha256\']:raise ValueError(\'Metric changed\')\n refs=identity[\'references\'];dev=run/\'development\'/identity[\'config\'][\'development_name\']\n for p,k in [(run/\'identity.json\',\'identity_sha256\'),(dev/\'identity.json\',\'step4_identity_sha256\'),(dev/\'cell_roles.parquet\',\'roles_sha256\'),(Path(common[\'outer_pair_contract\']),\'pair_sha256\')]:\n  if sha(p)!=refs[k]:raise ValueError(\'Source changed: \'+str(p))\n for p,h in [(Path(identity[\'target_history\'][\'prior_summary\']),identity[\'target_history\'][\'prior_summary_sha256\']),(run/\'step5\'/identity[\'config\'][\'prior_experiment\']/\'frozen_model_registry.json\',identity[\'target_history\'][\'prior_registry_sha256\'])]:\n  if sha(p)!=h:raise ValueError(\'Step5 history changed\')\n rp=run/\'step5\'/identity[\'config\'][\'prior_experiment\']/\'experiment_contract.json\';rc=js(rp)[\'config\']\n fields=[\'seed\',\'svd_components\',\'knn_k\',\'target_cells_per_metacell\',\'min_metacells\',\'max_metacells\',\'min_members\']\n representation={\'parameters\':{k:rc[k] for k in fields},\'source_sha256\':sha(rp),\n \'training_source\':\'T only for development; full A only for final refit\',\n \'algorithm\':\'Step5 SVD cosine kNN and normalized-membership Galerkin operator; exact shared IDs/hash for M0-M4; cache once per scope/phase\',\n \'Lc_audit\':\'must retain raw pre-spectral-normalization Galerkin operator, degree and weighted-degree vectors, eigenvalue maximum and total affinity weights; symmetrization and within-metacell-edge removal recorded\',\n \'fallback\':\'no per-method backend/assignment change; any global representation amendment requires a new version before search\'}\n grid=make_grid(coarse,common,representation);grid[\'common_protocol_sha256\']=sha(old/\'common_protocol.json\')\n name=protocol_name+\'_grid_v1\';out=old/\'step2_parameter_grid\';cfg=root/\'configs\'/name\n out.mkdir(parents=True,exist_ok=True);cfg.mkdir(parents=True,exist_ok=True)\n registration={\'version\':\'task2-step2-v1\',\'source_protocol_sha256\':sha(old/\'protocol_identity.json\'),\'source_manifest_sha256\':sha(old/\'output_manifest.json\'),\'engine_sha256\':ENGINE_SHA,\'grid\':grid}\n freeze(out/\'grid_identity.json\',registration);freeze(out/\'parameter_grid.yaml\',grid);freeze(cfg/\'parameter_grid.yaml\',grid)\n current=root/\'configs/parameter_grid.yaml\'\n if current.exists() and js(current)!=grid:\n  if js(current)!=coarse:raise ValueError(\'Root grid differs from Step2.1 coarse or this final grid\')\n  archive=root/\'configs\'/name/\'parameter_grid.step2_1_coarse.json\'\n  if archive.exists() and sha(archive)!=sha(current):raise ValueError(\'Coarse archive changed\')\n  if not archive.exists():shutil.copyfile(current,archive)\n save(current,grid)\n budgets=pd.DataFrame([{\'method\':m,**v} for m,v in grid[\'budget\'].items() if isinstance(v,dict)]);budgets.to_csv(out/\'search_budget.csv\',index=False)\n graph_summary=[]\n if do_graph_audit:\n  edgepath=root/\'data/processed/ppi_annotation_reference_edges.parquet\'\n  if sha(edgepath)!=refs[\'ppi_reference_sha256\']:raise ValueError(\'PPI changed\')\n  edges=pd.read_parquet(edgepath)\n  universes={\'outer\':common[\'outer_gene_universe\'],**common[\'inner_gene_universe\']}\n  for label,ref in universes.items():\n   path=Path(ref[\'path\'])\n   if sha(path)!=ref[\'sha256\']:raise ValueError(\'Gene universe changed\')\n   genes=pd.read_parquet(path).sort_values(\'selected_col\');symbols=genes.gene_symbol.tolist()\n   if len(set(symbols))!=len(symbols) or not np.array_equal(genes.selected_col,np.arange(len(genes))):raise ValueError(\'Invalid gene mapping\')\n   mapping={g:i for i,g in enumerate(symbols)};e=edges.loc[edges.gene_a.isin(mapping)&edges.gene_b.isin(mapping)]\n   uv=np.column_stack([e.gene_a.map(mapping),e.gene_b.map(mapping)]).astype(np.int64);w=e.ppi_weight.to_numpy(float)\n   if not len(w) or (w<=0).any():raise ValueError(\'Positive base edges required\')\n   uv,ws,audit=mc.normalize_base(uv,w,len(genes));freeze(out/f\'{label}_base_normalization.json\',audit)\n   before=audit[\'before\'];after=audit[\'after\']\n   pd.DataFrame({\'gene_symbol\':symbols,\'degree_before\':before[\'degree_sequence\'],\'degree_after\':after[\'degree_sequence\'],\'weighted_degree_before\':before[\'weighted_degree\'],\'weighted_degree_after\':after[\'weighted_degree\']}).to_csv(out/f\'{label}_node_degrees.csv\',index=False)\n   if not np.isclose(after[\'lambda_max\'],1.,rtol=1e-7):raise ValueError(\'Base normalization failed\')\n   graph_summary.append({\'universe\':label,\'n_genes\':len(genes),\'n_edges\':len(uv),\'gene_sha256\':ref[\'sha256\'],\'lambda_max_before\':before[\'lambda_max\'],\'lambda_max_after\':after[\'lambda_max\'],\'weight_sum_before\':before[\'weight_sum\'],\'weight_sum_after\':after[\'weight_sum\'],\'trace_Lg_before\':before[\'trace_Lg\'],\'trace_Lg_after\':after[\'trace_Lg\']})\n  pd.DataFrame(graph_summary).to_csv(out/\'base_graph_summary.csv\',index=False)\n freeze(out/\'normalization_pending.json\',{\'Lc\':\'raw and normalized degree/eigenvalue/weights must be saved when constructing shared Lc in 2.3\',\'M4\':\'each actual randomized base graph audited before search in 2.3\',\'M2_M3\':\'derived graph statistics/conditional EDF after actual fit, no separate spectral rescaling\'})\n freeze(out/\'precision_gate.json\',{\'status\':\'PENDING_REAL_T_V_PROBES_IN_2.3\',\'rules\':grid[\'precision\'],\'synthetic_PASS_is_not_real_PASS\':True})\n summary={\'status\':\'complete_parameter_search_contract_frozen_NOT_BENCHMARKED\',\'output_path\':str(out),\'config_path\':str(current),\'M1_M4_equal_nominal_unique_budget\':grid[\'budget\'][\'M1\'][\'nominal_unique\'],\n \'base_universes_audited\':len(graph_summary),\'new_B_read\':False,\'real_models_fitted\':False,\'real_precision_gate\':\'PENDING_2.3\',\'old_lock_before\':oldlock,\'old_lock_after\':sha(lock),\'next\':\'2.3 first four exploratory contexts; real T/V numerical gate before search\'}\n assert summary[\'old_lock_before\']==summary[\'old_lock_after\']\n freeze(out/\'task2_step2_summary.json\',summary)\n for p in [out/\'parameter_grid.yaml\',out/\'grid_identity.json\']:freeze(cfg/p.name,js(p))\n save(out/\'output_manifest.json\',{\'files\':[{\'name\':p.name,\'sha256\':sha(p)} for p in sorted(out.iterdir()) if p.is_file() and p.name!=\'output_manifest.json\']})\n return out,grid,summary\n'
module_path=ROOT/"src/parameter_search.py";module_path.parent.mkdir(parents=True,exist_ok=True)
if module_path.exists() and module_path.read_bytes()!=SOURCE.encode("utf-8"):
    old_sha=hashlib.sha256(module_path.read_bytes()).hexdigest()
    backup=module_path.with_name("parameter_search."+old_sha[:12]+".backup.py")
    if not backup.exists():backup.write_bytes(module_path.read_bytes())
module_path.write_text(SOURCE,encoding="utf-8")
ps=load_module("parameter_search",module_path)
print("2.2 source SHA256:",ps.ENGINE_SHA)


2.2 source SHA256: b7136338afcf3b85e9f75ebafa41f6439f525cd7c1a22af5d13b7b6c03cb5d28


In [4]:
TEST_SOURCE = "from pathlib import Path\nimport json,tempfile,sys\nimport numpy as np\nimport pandas as pd\nimport parameter_search as ps\nimport randomized_controls as m\n\ndef reject(f):\n try:f()\n except (ValueError,RuntimeError,KeyError):return\n raise AssertionError('Expected rejection')\n\ndef grid_tests():\n c=m.default_config(Path('/unused'));g=ps.make_grid(m.grid_contract(c),{'scoring':{'q':.5}})\n checks=[];counts={}\n for method in ['M0','M1','M2','M4']:\n  coarse=ps.candidates(method,g)\n  assert len(coarse)==g['budget'][method]['coarse']\n  for best in coarse:\n   fine=ps.candidates(method,g,'fine',best)\n   assert len(fine)==g['budget'][method]['fine_slots']\n   new={ps.candidate_id(x) for x in fine}-{ps.candidate_id(x) for x in coarse}\n   assert len(new)==g['budget'][method]['new_fine_fits']\n   for axis in ['alpha','beta']:\n    if method=='M0' and axis=='beta':continue\n    values=sorted(set(x[axis] for x in fine));a=g['coarse'][axis]\n    assert values[0]==0 and len(values)==6\n    assert a[1]<=values[1]<=values[-1]<=a[-1]\n   assert any(x['beta']==0 for x in fine)\n  counts[method]=g['budget'][method]['nominal_unique']\n assert ps.candidates('M3',g)==[]\n checks.append('Enumerated every coarse center, incl zero/bounds/fixed and finite lambda; exact constant fine/unique budgets')\n def callback(x):return {'status':'OK','target_role':'development_only','development_deviance':float((np.log10(max(x['alpha'],1e-6))+.2)**2+(np.log10(max(x['beta'],1e-9))+1.2)**2)}\n for method in ['M0','M1','M2','M4']:\n  r=ps.search(method,g,callback)\n  assert r['unique_attempts']==g['budget'][method]['nominal_unique']\n  assert sum(x['cache_reuse'] for x in r['rows'])==1\n  assert r['selected']['development_deviance']<=min(x['development_deviance'] for x in r['rows'] if x['phase']=='coarse')\n reject(lambda:ps.search('M1',g,lambda x:{'status':'FAILED'}))\n reject(lambda:ps.search('M1',g,lambda x:{'status':'OK','target_role':'final_B','development_deviance':1.}))\n reject(lambda:ps.search('M1',g,lambda x:{'status':'OK','target_role':'development_only','development_deviance':float('nan')}))\n def failed(x):\n  if x['alpha']==0 and x['beta']>0:raise RuntimeError('fixture failure')\n  return callback(x)\n r=ps.search('M4',g,failed);assert any(x['status']=='FAILED' for x in r['rows'])\n checks.append('Toy coarse/fine searches retain coarse optimum/failures, cache origin; all-invalid/NaN/final-B selection rejected')\n coarse=ps.candidates('M2',g);tie=[{**x,'status':'OK','target_role':'development_only','development_deviance':1.} for x in coarse]\n assert ps.winner(tie,coarse)['alpha']==ps.winner(tie,coarse)['beta']==0\n reject(lambda:ps.winner(tie[:-1],coarse))\n lr=g['source_coarse']['lambda_rules'];e=np.array([0.,1.,2.,5.,10.])\n assert ps.threshold('q0.5',e,lr)==3.5\n assert ps.threshold('scaled:q0.5:0.5',e,lr)==1.75\n assert ps.threshold('q0.5',np.zeros(5),lr)=='DEGENERATE_NO_POSITIVE_ENERGY'\n reject(lambda:ps.fine_axis([0.,1.],0.))\n checks.append('Deterministic ties, incomplete-table rejection, lambda training-threshold conversion and all-zero training energies')\n return {'status':'PASS','checks':checks,'nominal_unique_budgets':counts}\n\ndef numerical_tests():\n rng=np.random.default_rng(34);C=rng.normal(size=(4,8));lc=np.array([0.,.1,.5,1.])\n uv=np.array([[i,i+1] for i in range(7)],int);L=m.graph_laplacian(uv,np.arange(1.,8.),8)\n vals,U=np.linalg.eigh(L.toarray());tests=[]\n for alpha,beta in [(0.,0.),(.001,1e-6),(1.,1.),(100.,100.)]:\n  oracle=((C@U)/(1+alpha*lc[:,None]+beta*vals[None,:]))@U.T\n  x,res=ps.solve(C,lc,L,alpha,beta);tight,_=ps.solve(C,lc,L,alpha,beta,1e-10,3000)\n  assert np.allclose(tight,oracle,rtol=1e-7,atol=1e-9)\n  g=ps.make_grid(m.grid_contract(m.default_config(Path('/unused'))),{'scoring':{}})\n  # Synthetic positive count Poisson deviance, all zeros included.\n  B=rng.poisson(2.,size=C.shape).astype(float)\n  def loss(p):\n   from scipy.special import xlogy\n   mu=np.exp(np.clip(p,-10,10));return float(np.mean(2*(xlogy(B,B/mu)-B+mu)))\n  r=ps.precision_comparison(x,tight,loss,g['precision']['thresholds']);assert r['status']=='PASS';tests.append({'alpha':alpha,'beta':beta,'residual':res,**r})\n x,res=ps.solve(np.zeros_like(C),lc,L,100.,100.);assert np.count_nonzero(x)==0 and res==0\n reject(lambda:ps.solve(C,lc,L,100.,100.,1e-14,1))\n reject(lambda:ps.solve(C,lc,-L,1.,1.))\n reject(lambda:ps.solve(C,lc,L,1.,1.,rtol=0))\n reject(lambda:ps.solve(np.empty((0,8)),np.array([]),L,0.,0.))\n assert ps.precision_comparison(np.ones((2,2)),np.zeros((2,2)),lambda x:1.,g['precision']['thresholds'])['status']=='FAIL'\n reject(lambda:ps.precision_comparison(C,C,lambda x:float('nan'),g['precision']['thresholds']))\n return {'status':'PASS','checks':['CG vs independent eigendecomposition at zero/low/middle/high parameters','zero RHS, solver failure and invalid Laplacian/empty inputs','precision gate catches altered prediction and invalid loss'],'probes':tests}\n\ndef synthetic_project(root):\n run=root/'data/splits/step2_fixture';prep=run/'A_preprocessing';dev=run/'development/inner_q05_seed20261008_v1';prior=run/'step5/step5_four_contexts_v1'\n for p in [prep,dev,prior,run/'evaluation',root/'data/processed']:p.mkdir(parents=True,exist_ok=True)\n cells=pd.DataFrame({'matrix_row':np.arange(31),'cell_barcode':[f'c{i}' for i in range(31)],'evaluation_eligible':[True]*30+[False],\n 'cell_line':pd.Series([l for l in m.LINES for d in m.DRUGS for r in m.REPS]+[pd.NA],dtype='string'),\n 'drug':pd.Series([d for l in m.LINES for d in m.DRUGS for r in m.REPS]+[pd.NA],dtype='string'),\n 'replicate':pd.Series([r for l in m.LINES for d in m.DRUGS for r in m.REPS]+[pd.NA],dtype='string')})\n genes=pd.DataFrame({'selected_col':[0,1,2],'gene_idx':[1,2,3],'gene_symbol':['a','b','c']});cp=prep/'evaluation_cells.parquet';gp=prep/'selected_genes_A.parquet';cells.to_parquet(cp,index=False);genes.to_parquet(gp,index=False)\n pair={'cells_file':cp.name,'genes_file':gp.name,'cells_sha256':m.digest(cp),'genes_sha256':m.digest(gp),'n_cells':30,'n_genes':3,'n_pairs':90,'q':.5,'normalization_scale':10000}\n m.save(prep/'evaluation_pair_contract.json',pair);metric={**pair,'pair_contract_sha256':m.digest(prep/'evaluation_pair_contract.json'),'mu_floor':1e-8,'relative_min_denominator':1e-12,'primary':'mean Poisson deviance','secondary':'normalized MSE'};m.save(run/'evaluation/metric_contract.json',metric);m.save(run/'identity.json',{'seed':20261008})\n cells.to_parquet(dev/'cell_roles.parquet',index=False);inner=[]\n for r in m.REPS:\n  p=dev/f'{r}_inner_selected_genes.parquet';genes.to_parquet(p,index=False);inner.append({'replicate':r,'selected_genes_sha256':m.digest(p)})\n m.save(dev/'step4_summary.json',{'status':'step4_prepared','metric_contract_sha256_after':m.digest(run/'evaluation/metric_contract.json'),'inner_genes_by_replicate':inner})\n m.save(dev/'scenario_A_contract.json',{'inner_q':.5});m.save(dev/'identity.json',{'status':'frozen'})\n m.save(prior/'step5_summary.json',{'status':'four_context_baseline_scored','final_B_accessed':True,'contexts':m.FOUR});m.save(prior/'frozen_model_registry.json',{'status':'frozen'});m.save(run/'evaluation/target_access_lock.json',{'status':'already_accessed_keep_me'})\n pd.DataFrame({'gene_a':['a','b'],'gene_b':['b','c'],'ppi_weight':[.7,.8]}).to_parquet(root/'data/processed/ppi_annotation_reference_edges.parquet',index=False)\n # Poison target and old-score files: preparation must not read them.\n (run/'B_target.npz').write_bytes(b'POISON');(prior/'baseline_reproduction.csv').write_bytes(b'POISON')\n c=m.default_config(root);c['run_path']=str(run);return c\n\ndef integration_tests():\n with tempfile.TemporaryDirectory() as tmp:\n  root=Path(tmp);cfg=synthetic_project(root);run=Path(cfg['run_path']);old,_=m.prepare_contract(cfg)\n  ps.save(run/'step5'/cfg['prior_experiment']/'experiment_contract.json',{'config':{'seed':20261008,'svd_components':50,'knn_k':30,'target_cells_per_metacell':1050,'min_metacells':12,'max_metacells':80,'min_members':30}})\n  prior_bytes=(old/'parameter_grid_coarse.json').read_bytes();lock=(run/'evaluation/target_access_lock.json').read_bytes()\n  out,g,s=ps.prepare(root,run,cfg['protocol_name'],m)\n  assert s['base_universes_audited']==3 and not s['new_B_read'] and not s['real_models_fitted']\n  assert (old/'parameter_grid_coarse.json').read_bytes()==prior_bytes\n  assert (run/'evaluation/target_access_lock.json').read_bytes()==lock\n  assert ps.js(root/'configs/parameter_grid.yaml')==g\n  assert ps.js(root/'configs'/cfg['protocol_name']/'parameter_grid.yaml')['status'].startswith('coarse')\n  assert ps.prepare(root,run,cfg['protocol_name'],m)[2]==s\n  for r in ps.js(out/'output_manifest.json')['files']:assert ps.sha(out/r['name'])==r['sha256']\n  assert ps.js(out/'precision_gate.json')['status']=='PENDING_REAL_T_V_PROBES_IN_2.3'\n  # Reject root-grid changes and mutated source.\n  ps.save(root/'configs/parameter_grid.yaml',{'bad':'changed'})\n  reject(lambda:ps.prepare(root,run,cfg['protocol_name'],m))\n  ps.save(root/'configs/parameter_grid.yaml',g)\n  (run/'evaluation/target_access_lock.json').write_text('{}')\n  reject(lambda:ps.prepare(root,run,cfg['protocol_name'],m))\n return {'status':'PASS','checks':['Full synthetic project: three base-universe normalization audits and immutable retry','Coarse archive preserved, root YAML promoted, old lock unchanged','Manifest, pending real precision gate, changed config/source rejection']}\n\ndef run_tests():return {'grids':grid_tests(),'numerical':numerical_tests(),'integration':integration_tests()}\n\nif __name__=='__main__':print(json.dumps(run_tests(),indent=2))\n"
test_ns={"__name__":"task2_step2_tests","__file__":str(ROOT/"tests/test_parameter_search.py")}
exec(compile(TEST_SOURCE,"embedded_parameter_tests.py","exec"),test_ns)
TEST_REPORT=test_ns["run_tests"]()
assert all(x["status"]=="PASS" for x in TEST_REPORT.values())
print(json.dumps(TEST_REPORT,ensure_ascii=False,indent=2))


{
  "grids": {
    "status": "PASS",
    "checks": [
      "Enumerated every coarse center, incl zero/bounds/fixed and finite lambda; exact constant fine/unique budgets",
      "Toy coarse/fine searches retain coarse optimum/failures, cache origin; all-invalid/NaN/final-B selection rejected",
      "Deterministic ties, incomplete-table rejection, lambda training-threshold conversion and all-zero training energies"
    ],
    "nominal_unique_budgets": {
      "M0": 12,
      "M1": 105,
      "M2": 663,
      "M4": 105
    }
  },
  "numerical": {
    "status": "PASS",
    "checks": [
      "CG vs independent eigendecomposition at zero/low/middle/high parameters",
      "zero RHS, solver failure and invalid Laplacian/empty inputs",
      "precision gate catches altered prediction and invalid loss"
    ],
    "probes": [
      {
        "alpha": 0.0,
        "beta": 0.0,
        "residual": 0.0,
        "status": "PASS",
        "relative_prediction_l2": 0.0,
        "relative_development_

In [5]:
OUTPUT,GRID,SUMMARY=ps.prepare(ROOT,RUN,PROTOCOL_NAME,mc,do_graph_audit=True)
ps.save(OUTPUT/"synthetic_test_report.json",TEST_REPORT)
manifest=ps.js(OUTPUT/"output_manifest.json")
manifest["files"]=[r for r in manifest["files"] if r["name"]!="synthetic_test_report.json"]
manifest["files"].append({"name":"synthetic_test_report.json","sha256":ps.sha(OUTPUT/"synthetic_test_report.json")})
ps.save(OUTPUT/"output_manifest.json",manifest)
print(json.dumps(SUMMARY,ensure_ascii=False,indent=2))


{
  "status": "complete_parameter_search_contract_frozen_NOT_BENCHMARKED",
  "output_path": "/home/featurize/work/biological_prior_validation/data/splits/step2_q05_seed20261008/task2_protocols/task2_five_methods/step2_parameter_grid",
  "config_path": "/home/featurize/work/biological_prior_validation/configs/parameter_grid.yaml",
  "M1_M4_equal_nominal_unique_budget": 105,
  "base_universes_audited": 3,
  "new_B_read": false,
  "real_models_fitted": false,
  "real_precision_gate": "PENDING_2.3",
  "old_lock_before": "c166f83b5ff000156fb56c7f6c39084086f89d439fd5fd9396a3735d9bd6d9fe",
  "old_lock_after": "c166f83b5ff000156fb56c7f6c39084086f89d439fd5fd9396a3735d9bd6d9fe",
  "next": "2.3 first four exploratory contexts; real T/V numerical gate before search"
}


In [6]:
budgets=pd.read_csv(OUTPUT/"search_budget.csv")
assert budgets.set_index("method").loc["M1","nominal_unique"]==budgets.set_index("method").loc["M4","nominal_unique"]
print(budgets.to_string(index=False))
print("alpha coarse:",GRID["coarse"]["alpha"])
print("beta coarse:",GRID["coarse"]["beta"])
for center in [0.,1.,100.]:
    print("Example center",center,"alpha fine:",GRID["fine"]["alpha_lookup"][ps.key(center)])
    print("Example center",center,"beta fine:",GRID["fine"]["beta_lookup"][ps.key(center)])
print(pd.read_csv(OUTPUT/"base_graph_summary.csv").to_string(index=False))
print("Real accuracy gate:",ps.js(OUTPUT/"precision_gate.json")["status"])
print("Shared construction:",json.dumps(GRID["shared_representation"],ensure_ascii=False,indent=2))


method  coarse  fine_slots  new_fine_fits  nominal_unique
    M0       7           6              5              12
    M1      70          36             35             105
    M2     448         216            215             663
    M3       0           0              0               0
    M4      70          36             35             105
alpha coarse: [0.0, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
beta coarse: [0.0, 1e-06, 1e-05, 0.0001, 0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
Example center 0.0 alpha fine: [0.0, 0.0014683621015645027, 0.0021560872613109227, 0.0031622776601683794, 0.0046380312056200825, 0.006810309248206047]
Example center 0.0 beta fine: [0.0, 1.4683621015645011e-06, 2.1560872613109228e-06, 3.162277660168379e-06, 4.638031205620082e-06, 6.810309248206055e-06]
Example center 1.0 alpha fine: [0.0, 0.21560872613109228, 0.46380312056200823, 1.0023029324930393, 2.1560872613109225, 4.6380312056200825]
Example center 1.0 beta fine: [0.0, 0.21560872613109228, 0.46380312056200823

In [7]:
assert not SUMMARY["new_B_read"] and not SUMMARY["real_models_fitted"]
assert SUMMARY["old_lock_before"]==SUMMARY["old_lock_after"]
assert ps.js(ROOT/"configs/parameter_grid.yaml")==GRID
for r in ps.js(OUTPUT/"output_manifest.json")["files"]:
    assert ps.sha(OUTPUT/r["name"])==r["sha256"],r["name"]
REVIEW_ZIP=ROOT/"results/summary/task2_step2_review_bundle.zip"
REVIEW_ZIP.parent.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(REVIEW_ZIP,"w",zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUTPUT.iterdir()):
        if p.is_file():z.write(p,p.name)
    z.write(ROOT/"configs/parameter_grid.yaml","configs/parameter_grid.yaml")
    z.write(ROOT/"src/parameter_search.py","src/parameter_search.py")
with zipfile.ZipFile(REVIEW_ZIP) as z:assert z.testzip() is None
print("2.2 Configuration and basic PPI check completed, please return:",REVIEW_ZIP)
print("Next step 2.3: First carry out the real T/V numerical gate, then run the four exploratory contexts according to the frozen rules.")


2.2 Configuration and basic PPI check completed, please return: /home/featurize/work/biological_prior_validation/results/summary/task2_step2_review_bundle.zip
Next step 2.3: First carry out the real T/V numerical gate, then run the four exploratory contexts according to the frozen rules.
